In [1]:
import os
cwd = os.getcwd().split("\\")[:-1]
cwd = "\\".join(cwd)
os.chdir(cwd)
print(cwd)

C:\Users\Local_Admin\projects\h_lacustris


In [2]:
import warnings
# Suppress all warnings
warnings.filterwarnings("ignore")

import numpy as np
import polars as pl

from lmfit import Model
from tqdm import tqdm

In [3]:
def load_biolog_data(
    files: list[str], plate_idx, repl_str, repl_end
) -> pl.DataFrame:
    plate_key = {"1": "PM1", "2": "PM2", "N": "PM3"}
    n_wells = 96
    col_letters = "A:AK"
    
    # Get all the data
    start_row = 30
    read_options = {
        "header_row": start_row,
        "n_rows": n_wells,
        "use_columns": col_letters,
    }
    df_dict = pl.read_excel(
        file,
        sheet_id=0,
        read_options=read_options,
        has_header=True,
        infer_schema_length=5,
    )
    for k, v in df_dict.items():
        df_dict[k] = v.rename({"Wavel.": "well"})
    
    # Get all dates
    date_col = "B"
    start_row = 26
    read_options = {
        "skip_rows": start_row,
        "n_rows": 1,
        "use_columns": date_col,
    }
    dates = pl.read_excel(file, sheet_id=0, read_options=read_options)
    
    # Annotate the data with replicate and plate info
    for sheet, v in df_dict.items():
        try:
            plate = plate_key[sheet[plate_idx]]
        except KeyError:
            print(sheet)
            raise KeyError
        replicate = sheet[repl_str:repl_end]
        dates[sheet] = dates[sheet].with_columns(
                    pl.col("__UNNAMED__1").str.to_datetime("%m/%d/%Y %I:%M:%S %p"),
        )
        df_dict[sheet] = df_dict[sheet].with_columns(
            plate=pl.lit(plate),
            replicate=pl.lit(replicate),
            date=dates[sheet][0, 0]
        )
    
    od_df = pl.concat(df_dict.values()).sort("date")
    od_df = od_df.with_columns(
            time = pl.col("date") - od_df[0, -1],
        )
    
    return od_df.with_columns(time = pl.col("time").dt.total_seconds() / (3600 * 24))

def gompertz(t, A, B, C):
    """Get gompertz model."""
    return A * np.exp(-np.exp((B * np.e / A) * (C - t) + 1))

def fit_gompertz(dataframe):
    """Fit data to gompertz model.

    Parameters
    ----------
    t: time data
    y: growth data
    A, B, C: Initial conditions for gompertz Parameters

    """
    # Initial conditions
    A=1.0
    B=0.03
    C=3
    y1 = dataframe["680"].to_numpy()
    t1 = dataframe["time"].to_numpy()
    dataframe = dataframe.drop_nans(subset="logratio")
    y2 = dataframe["logratio"].to_numpy()
    t2 = dataframe["time"].to_numpy()
    model = Model(gompertz)
    params = model.make_params(A=A, B=B, C=C)
    params["A"].min = 0.001
    params["B"].min = 0
    params["C"].min = 0
    params["A"].max = 100
    params["B"].max = 2
    params["C"].max = 100
    #result1 = model.fit(y1, params, t=t1, method="differential_evolution")
    try:
        result2 = model.fit(y2, params, t=t2, method="differential_evolution")
        result_dict = {
            "model": "gompertz",
            "mu": result2.params["B"].value,
            "r2": result2.rsquared
        }
    except TypeError:
        result_dict = {
            "model": "gompertz",
            "mu": np.nan,
            "r2": np.nan
        }
    except AttributeError:
        result_dict = {
            "model": "gompertz",
            "mu": np.nan,
            "r2": np.nan
        }
    except RuntimeError:
        result_dict = {
            "model": "gompertz",
            "mu": np.nan,
            "r2": np.nan
        }
    
    return result_dict

def fit_log(dataframe):
    """Fit data to exponetial model.

    Parameters
    ----------
    t: time data
    y: growth data
    m: growth rate
    """
    def linear_model(t, m, b):
        return m * t + b
    # Initial guess
    m=0.03
    b=0.05
    # Log transform the data
    y = np.log(dataframe["680"].to_numpy())
    t = dataframe["time"].to_numpy()
    model = Model(linear_model)
    params = model.make_params(m=m, b=b)
    params["m"].min = 0
    result = model.fit(y, params, t=t)

    return {
        "model": "log",
        "mu": result.params["m"].value,
        "r2": result.rsquared
    }
    

### Load the data
____

In [4]:
# Light condition
file = "data/0_raw/optical_density/E250125.xlsx"
light = load_biolog_data(file, -1, 3, 5)
light = light.with_columns(
    condition=pl.lit("light")
)
# Dark condition
file = "data/0_raw/optical_density/E250708.xlsx"
dark = load_biolog_data(file, -4, -2, None)
dark = dark.with_columns(
    condition=pl.lit("dark")
)
# Merge everyting
data_df = pl.concat([light, dark])
print(data_df.head())

shape: (5, 42)
┌──────┬────────┬────────┬────────┬───┬───────────┬─────────────────────┬──────┬───────────┐
│ well ┆ 300    ┆ 320    ┆ 340    ┆ … ┆ replicate ┆ date                ┆ time ┆ condition │
│ ---  ┆ ---    ┆ ---    ┆ ---    ┆   ┆ ---       ┆ ---                 ┆ ---  ┆ ---       │
│ str  ┆ f64    ┆ f64    ┆ f64    ┆   ┆ str       ┆ datetime[μs]        ┆ f64  ┆ str       │
╞══════╪════════╪════════╪════════╪═══╪═══════════╪═════════════════════╪══════╪═══════════╡
│ A1   ┆ 1.2635 ┆ 0.8237 ┆ 0.6228 ┆ … ┆ P1        ┆ 2025-01-25 11:42:35 ┆ 0.0  ┆ light     │
│ A2   ┆ 1.2591 ┆ 0.8166 ┆ 0.6175 ┆ … ┆ P1        ┆ 2025-01-25 11:42:35 ┆ 0.0  ┆ light     │
│ A3   ┆ 1.2654 ┆ 0.8355 ┆ 0.6504 ┆ … ┆ P1        ┆ 2025-01-25 11:42:35 ┆ 0.0  ┆ light     │
│ A4   ┆ 1.2669 ┆ 0.8211 ┆ 0.6122 ┆ … ┆ P1        ┆ 2025-01-25 11:42:35 ┆ 0.0  ┆ light     │
│ A5   ┆ 1.2324 ┆ 0.8154 ┆ 0.6186 ┆ … ┆ P1        ┆ 2025-01-25 11:42:35 ┆ 0.0  ┆ light     │
└──────┴────────┴────────┴────────┴───┴───────────┴────

### Correct OD with pathlenght
---

In [7]:
# Volume for each well
data_df = data_df.with_columns(
    volume=(pl.col("980") - pl.col("900"))/0.0010018,
)
# Pathlenght for each well
data_df = data_df.with_columns(
    pathlenght=(pl.col("volume") * 0.0055) + 0.0128
)
# Corrected OD dataframe
wavelenghts = ["440", "680"]
cols = ["well", "plate", "replicate", "date", "time", "condition", "volume", "pathlenght"]
corrected_df = data_df.select(cols+wavelenghts).with_columns(
    pl.col("440")/pl.col("pathlenght"),
    pl.col("680")/pl.col("pathlenght"),
)
corrected_df.head()

well,plate,replicate,date,time,condition,volume,pathlenght,440,680
str,str,str,datetime[μs],f64,str,f64,f64,f64,f64
"""A1""","""PM3""","""P1""",2025-01-25 11:42:35,0.0,"""light""",102.515475,0.576635,0.258742,0.201861
"""A2""","""PM3""","""P1""",2025-01-25 11:42:35,0.0,"""light""",101.916543,0.573341,0.262496,0.202846
"""A3""","""PM3""","""P1""",2025-01-25 11:42:35,0.0,"""light""",102.315838,0.575537,0.265144,0.204157
"""A4""","""PM3""","""P1""",2025-01-25 11:42:35,0.0,"""light""",102.615289,0.577184,0.25815,0.19803
"""A5""","""PM3""","""P1""",2025-01-25 11:42:35,0.0,"""light""",101.816735,0.572792,0.255241,0.196406


### Fit to gompertz and linear models
---

In [141]:
cols = ["condition", "plate", "well", "replicate"]
gpb = corrected_df.group_by("condition", "plate", "well", "replicate")
data_to_concat = []
fits_to_concat = []
for name, data in tqdm(gpb):
    # Apply log(od_t / od_time_zero) to each well
    # Get log ratio
    data_tf = data.with_columns(
        logratio=np.log(pl.col("680")/pl.col("680").first())
    )
    # Replace negative values with nan
    data_tf = data_tf.with_columns(
        logratio=pl.when(pl.col("logratio") < 0).then(np.nan).otherwise(pl.col("logratio"))
    )
    # Save transformed data
    data_to_concat.append(data_tf)

    # Fit to the gompertz model
    keys = dict(zip(cols, name))
    fit_results = fit_gompertz(data_tf)
    keys.update(fit_results)
    fits_to_concat.append(pl.DataFrame(keys))
    # Fit to the linear model
    keys = dict(zip(cols, name))
    fit_results_l = fit_log(data_tf)
    keys.update(fit_results_l)
    fits_to_concat.append(pl.DataFrame(keys))
    
corrected_df = pl.concat(data_to_concat)
fits_df = pl.concat(fits_to_concat)
fits_df

2016it [22:39,  1.48it/s]


condition,plate,well,replicate,model,mu,r2
str,str,str,str,str,f64,f64
"""dark""","""PM2""","""D12""","""P2""","""gompertz""",0.606949,0.985116
"""dark""","""PM2""","""D12""","""P2""","""log""",0.31453,0.915467
"""light""","""PM3""","""D6""","""P3""","""gompertz""",1.988969,0.972979
"""light""","""PM3""","""D6""","""P3""","""log""",0.025337,0.840075
"""light""","""PM3""","""B1""","""P2""","""gompertz""",0.188422,0.563285
…,…,…,…,…,…,…
"""light""","""PM1""","""D11""","""P1""","""log""",0.185288,0.670662
"""light""","""PM1""","""F10""","""P3""","""gompertz""",0.251164,0.960663
"""light""","""PM1""","""F10""","""P3""","""log""",0.150992,0.927361


In [144]:
corrected_df.write_csv("data/2_processed/corrected_od.csv")
fits_df.write_csv("data/2_processed/growth_rates.csv")
print("Data saved to: data/2_processed/")

Data saved to: data/2_processed
